# Mini-Lab D: Traffic Splitting — Champion/Challenger Deployment

**Goal:** Deploy two model versions to a single Vertex AI endpoint with traffic splitting, verify the split via prediction request logging to BigQuery, and simulate a canary rollout.

**Dataset:** Census income (same as Labs 1–4, Mini-Lab A)

**Key Exam Concepts:**
- Traffic splitting on Vertex AI endpoints
- Canary vs A/B vs shadow deployment strategies
- Prediction request-response logging
- Gradual rollout patterns

**Container:** Prebuilt `sklearn-cpu.1-3:latest` for serving

---

## Part 1: Setup + Data Prep

In [1]:
# Standard imports
import os
import json
import time
import pickle
import tempfile
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from google.cloud import bigquery, storage, aiplatform

print("Imports complete.")

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv-tf/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/james.carty/Documents/VScode/google-ml-engineer/.venv-tf/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureW

Imports complete.


In [2]:
# Project configuration
PROJECT_ID = "carty-470812"
REGION = "us-central1"
BUCKET_NAME = "carty-470812"
BUCKET_URI = f"gs://{BUCKET_NAME}"

# Lab-specific paths
LAB_PREFIX = "mini-lab-d"
MODEL_DIR = f"{BUCKET_URI}/{LAB_PREFIX}/models"

# BigQuery logging dataset
BQ_LOG_DATASET = "prediction_logs"

# Initialize clients
bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

print(f"Project: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Model artifacts: {MODEL_DIR}")
print(f"BQ logging dataset: {BQ_LOG_DATASET}")

Project: carty-470812
Region: us-central1
Model artifacts: gs://carty-470812/mini-lab-d/models
BQ logging dataset: prediction_logs


In [3]:
# Pull census data from BigQuery
query = """
SELECT
    age,
    workclass,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income_bracket
FROM `bigquery-public-data.ml_datasets.census_adult_income`
WHERE workclass IS NOT NULL
  AND occupation IS NOT NULL
  AND native_country IS NOT NULL
"""

df = bq_client.query(query).to_dataframe()
print(f"Dataset shape: {df.shape}")
df.head()

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv-tf/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Dataset shape: (32561, 14)


,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income_bracket
0,39,Private,9th,5,Married-civ-spouse,Other-service,Wife,Black,Female,3411,0,34,United-States,<=50K
1,77,Private,9th,5,Married-civ-spouse,Priv-house-serv,Wife,Black,Female,0,0,10,United-States,<=50K
2,38,Private,9th,5,Married-civ-spouse,Other-service,Wife,Black,Female,0,0,24,Haiti,<=50K
3,28,Private,9th,5,Married-civ-spouse,Protective-serv,Wife,Black,Female,0,0,40,United-States,<=50K
4,37,Private,9th,5,Married-civ-spouse,Machine-op-inspct,Wife,Black,Female,0,0,48,United-States,<=50K


In [4]:
# Preprocess: encode categoricals, create target
# Binary target
df["target"] = (df["income_bracket"].str.strip() == ">50K").astype(int)
df = df.drop(columns=["income_bracket"])

# Identify column types
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = [c for c in df.columns if c not in categorical_cols and c != "target"]

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")

Categorical columns (8): ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']
Numerical columns (5): ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']


In [5]:
# Label encode categoricals
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Train/test split — hash-based for reproducibility
np.random.seed(42)
mask = np.random.rand(len(df)) < 0.8
train_df = df[mask].copy()
test_df = df[~mask].copy()

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]
X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).to_dict()}")

Train: (26070, 13), Test: (6491, 13)
Target distribution (train): {0: 0.7602224779439969, 1: 0.23977752205600306}


---
## Part 2: Train Two Model Versions Locally

We train two Random Forest classifiers with different hyperparameters:
- **Champion:** Conservative — fewer trees, shallower depth (the "current production" model)
- **Challenger:** Aggressive — more trees, deeper (the model we think might be better)

We evaluate both locally first so we know the ground truth before deploying.

In [6]:
# Train Champion: conservative hyperparameters
champion_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
champion_model.fit(X_train, y_train)

champion_preds = champion_model.predict(X_test)
print("=== Champion Model (n_estimators=100, max_depth=10) ===")
print(f"Accuracy:  {accuracy_score(y_test, champion_preds):.4f}")
print(f"Precision: {precision_score(y_test, champion_preds):.4f}")
print(f"Recall:    {recall_score(y_test, champion_preds):.4f}")
print(f"F1:        {f1_score(y_test, champion_preds):.4f}")

=== Champion Model (n_estimators=100, max_depth=10) ===
Accuracy:  0.8566
Precision: 0.8111
Recall:    0.5403
F1:        0.6485


In [7]:
# Train Challenger: more aggressive hyperparameters
challenger_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
challenger_model.fit(X_train, y_train)

challenger_preds = challenger_model.predict(X_test)
print("=== Challenger Model (n_estimators=300, max_depth=20) ===")
print(f"Accuracy:  {accuracy_score(y_test, challenger_preds):.4f}")
print(f"Precision: {precision_score(y_test, challenger_preds):.4f}")
print(f"Recall:    {recall_score(y_test, challenger_preds):.4f}")
print(f"F1:        {f1_score(y_test, challenger_preds):.4f}")

=== Challenger Model (n_estimators=300, max_depth=20) ===
Accuracy:  0.8617
Precision: 0.7742
Recall:    0.6145
F1:        0.6851


In [9]:
# Compare side-by-side
print("=== Local Evaluation Comparison ===")
print(f"{'Metric':<12} {'Champion':>10} {'Challenger':>12}")
print("-" * 36)
for metric_name, metric_fn in [
    ("Accuracy", accuracy_score),
    ("Precision", precision_score),
    ("Recall", recall_score),
    ("F1", f1_score),
]:
    c = metric_fn(y_test, champion_preds)
    ch = metric_fn(y_test, challenger_preds)
    better = "<--" if ch > c else ""
    print(f"{metric_name:<12} {c:>10.4f} {ch:>12.4f}  {better}")

# How often do they disagree?
disagreements = (champion_preds != challenger_preds).sum()
print(f"\nPrediction disagreements: {disagreements}/{len(y_test)} ({disagreements/len(y_test)*100:.1f}%)")

=== Local Evaluation Comparison ===
Metric         Champion   Challenger
------------------------------------
Accuracy         0.8566       0.8617  <--
Precision        0.8111       0.7742  
Recall           0.5403       0.6145  <--
F1               0.6485       0.6851  <--

Prediction disagreements: 369/6491 (5.7%)


In [10]:
# Save models and upload to GCS
# Vertex AI expects model artifacts in a GCS directory
# sklearn models: save as model.pkl using pickle protocol 4 (for Python 3.10 serving container)

bucket = storage_client.bucket(BUCKET_NAME)

for name, model in [("champion", champion_model), ("challenger", challenger_model)]:
    # Save locally first
    local_path = f"{name}_model.pkl"
    with open(local_path, "wb") as f:
        pickle.dump(model, f, protocol=4)
    
    # Upload to GCS
    gcs_path = f"{LAB_PREFIX}/models/{name}/model.pkl"
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    print(f"Uploaded {name} model to gs://{BUCKET_NAME}/{gcs_path}")

print("\nBoth models uploaded to GCS.")

Uploaded champion model to gs://carty-470812/mini-lab-d/models/champion/model.pkl
Uploaded challenger model to gs://carty-470812/mini-lab-d/models/challenger/model.pkl

Both models uploaded to GCS.


In [11]:
# Register both as Vertex AI Model resources
SKLEARN_CONTAINER = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"

champion_vertex_model = aiplatform.Model.upload(
    display_name=f"{LAB_PREFIX}-champion",
    artifact_uri=f"{BUCKET_URI}/{LAB_PREFIX}/models/champion",
    serving_container_image_uri=SKLEARN_CONTAINER,
)
print(f"Champion model registered: {champion_vertex_model.resource_name}")

challenger_vertex_model = aiplatform.Model.upload(
    display_name=f"{LAB_PREFIX}-challenger",
    artifact_uri=f"{BUCKET_URI}/{LAB_PREFIX}/models/challenger",
    serving_container_image_uri=SKLEARN_CONTAINER,
)
print(f"Challenger model registered: {challenger_vertex_model.resource_name}")

Creating Model
Create Model backing LRO: projects/873708835509/locations/us-central1/models/4036895424286556160/operations/8128537808852746240
Model created. Resource name: projects/873708835509/locations/us-central1/models/4036895424286556160@1
To use this Model in another session:
model = aiplatform.Model('projects/873708835509/locations/us-central1/models/4036895424286556160@1')
Champion model registered: projects/873708835509/locations/us-central1/models/4036895424286556160
Creating Model
Create Model backing LRO: projects/873708835509/locations/us-central1/models/4745086465690566656/operations/1624214047022907392
Model created. Resource name: projects/873708835509/locations/us-central1/models/4745086465690566656@1
To use this Model in another session:
model = aiplatform.Model('projects/873708835509/locations/us-central1/models/4745086465690566656@1')
Challenger model registered: projects/873708835509/locations/us-central1/models/4745086465690566656


---
## Part 3: Deploy with Traffic Split + Prediction Logging

We deploy both models to **one endpoint** with an 80/20 traffic split.

We also enable **prediction request-response logging** to BigQuery. This writes every prediction request and response to a BQ table, including which deployed model served it — our key mechanism for verifying the split.

**Exam note:** Traffic splitting is the native Vertex AI mechanism for canary and A/B deployments. The `traffic_split` parameter is a dict mapping deployed model IDs to percentages that must sum to 100.

In [12]:
# Create the BigQuery dataset for prediction logs (if it doesn't exist)
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{BQ_LOG_DATASET}")
dataset_ref.location = REGION

try:
    bq_client.get_dataset(dataset_ref)
    print(f"Dataset {BQ_LOG_DATASET} already exists.")
except Exception:
    dataset = bq_client.create_dataset(dataset_ref)
    print(f"Created dataset: {dataset.dataset_id}")

Dataset prediction_logs already exists.


In [13]:
# Create endpoint with prediction request-response logging enabled
endpoint = aiplatform.Endpoint.create(
    display_name=f"{LAB_PREFIX}-endpoint",
    enable_request_response_logging=True,
    request_response_logging_bq_destination_table=f"bq://{PROJECT_ID}.{BQ_LOG_DATASET}",
)
print(f"Endpoint created: {endpoint.resource_name}")

Creating Endpoint
Create Endpoint backing LRO: projects/873708835509/locations/us-central1/endpoints/1246688405979398144/operations/3232210220226707456
Endpoint created. Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/873708835509/locations/us-central1/endpoints/1246688405979398144')
Endpoint created: projects/873708835509/locations/us-central1/endpoints/1246688405979398144


In [14]:
# Deploy champion first (gets 80% of traffic)
# We deploy both models in sequence and set the traffic split after

# Deploy champion with 100% initially (required for first deployment)
champion_vertex_model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="champion",
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=100,  # Takes all traffic initially
)
print("Champion deployed (100% traffic).")

Deploying model to Endpoint : projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Deploy Endpoint model backing LRO: projects/873708835509/locations/us-central1/endpoints/1246688405979398144/operations/8070835438627061760
Endpoint model deployed. Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Champion deployed (100% traffic).


In [15]:
# Now deploy challenger and set the 80/20 split
# When deploying a second model, we specify the traffic_split for ALL deployed models

# First, get the champion's deployed model ID
endpoint_info = endpoint.gca_resource
champion_deployed_id = endpoint_info.deployed_models[0].id
print(f"Champion deployed model ID: {champion_deployed_id}")

# Deploy challenger with traffic split
challenger_vertex_model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="challenger",
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1,
    traffic_split={champion_deployed_id: 80, "0": 20},  # "0" = this new deployment
)
print("Challenger deployed (80/20 split).")

Champion deployed model ID: 6624945135199191040
Deploying model to Endpoint : projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Deploy Endpoint model backing LRO: projects/873708835509/locations/us-central1/endpoints/1246688405979398144/operations/8092790486810492928
Endpoint model deployed. Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Challenger deployed (80/20 split).


In [16]:
# Verify the traffic split configuration
endpoint = aiplatform.Endpoint(endpoint.resource_name)  # Refresh

print("=== Deployed Models ===")
deployed_models = {}
for dm in endpoint.gca_resource.deployed_models:
    print(f"  ID: {dm.id}, Display: {dm.display_name}, Model: {dm.model}")
    deployed_models[dm.display_name] = dm.id

print(f"\n=== Traffic Split ===")
print(f"  {endpoint.gca_resource.traffic_split}")

champion_deployed_id = deployed_models["champion"]
challenger_deployed_id = deployed_models["challenger"]
print(f"\nChampion deployed ID:   {champion_deployed_id}")
print(f"Challenger deployed ID: {challenger_deployed_id}")

=== Deployed Models ===
  ID: 3685924165874876416, Display: challenger, Model: projects/873708835509/locations/us-central1/models/4745086465690566656
  ID: 6624945135199191040, Display: champion, Model: projects/873708835509/locations/us-central1/models/4036895424286556160

=== Traffic Split ===
  {'3685924165874876416': 20, '6624945135199191040': 80}

Champion deployed ID:   6624945135199191040
Challenger deployed ID: 3685924165874876416


---
## Part 4: Send Predictions + Verify Split via BigQuery Logs

We send a batch of prediction requests and then query the BigQuery prediction logs to verify which deployed model handled each request.

**Note:** Prediction logs take a few minutes to appear in BigQuery. We'll build in a retry loop.

In [17]:
# Prepare test instances — pick rows where the models disagree for more interesting analysis
# But also include some agreement rows for a realistic mix
test_instances = X_test.head(20).values.tolist()
print(f"Prepared {len(test_instances)} test instances.")
print(f"Features per instance: {len(test_instances[0])}")

Prepared 20 test instances.
Features per instance: 13


In [18]:
# Send predictions — 150 requests to get a good sample for verifying the split
# Each request sends a single instance so we can see per-request routing

NUM_REQUESTS = 150
predictions = []

print(f"Sending {NUM_REQUESTS} prediction requests...")
start_time = datetime.utcnow()

for i in range(NUM_REQUESTS):
    # Cycle through test instances
    instance = test_instances[i % len(test_instances)]
    response = endpoint.predict(instances=[instance])
    predictions.append(response.predictions[0])
    
    if (i + 1) % 50 == 0:
        print(f"  Sent {i + 1}/{NUM_REQUESTS}")

end_time = datetime.utcnow()
print(f"\nAll {NUM_REQUESTS} requests sent.")
print(f"Time range: {start_time.isoformat()} to {end_time.isoformat()}")

Sending 150 prediction requests...
  Sent 50/150
  Sent 100/150
  Sent 150/150

All 150 requests sent.
Time range: 2026-03-15T05:40:31.557814 to 2026-03-15T05:41:13.118756


In [19]:
# Quick look at prediction distribution (statistical check before logs arrive)
pred_counts = pd.Series(predictions).value_counts()
print("Prediction value distribution:")
print(pred_counts)
print(f"\nIf models disagree on some inputs, an uneven split suggests traffic routing is working.")

Prediction value distribution:
0.0    149
1.0      1
Name: count, dtype: int64

If models disagree on some inputs, an uneven split suggests traffic routing is working.


In [20]:
# Wait for prediction logs to appear in BigQuery
# Logs can take 2-10 minutes to flow through

# The table name is auto-created by Vertex AI in the format:
# prediction_logs.serving_predict_<endpoint_number>
# Let's discover it by listing tables in the dataset

print("Waiting for prediction logs to appear in BigQuery...")
print("(This can take 5-10 minutes. Logs are batched before writing.)")
print()

log_table = None
for attempt in range(20):  # Try for up to ~10 minutes
    tables = list(bq_client.list_tables(f"{PROJECT_ID}.{BQ_LOG_DATASET}"))
    if tables:
        log_table = tables[0]  # Should be the serving predict table
        print(f"Found log table: {log_table.full_table_id}")
        break
    else:
        print(f"  Attempt {attempt + 1}/20 — no tables yet, waiting 30s...")
        time.sleep(30)

if log_table is None:
    print("\n⚠️ Log table not found yet. This can take longer sometimes.")
    print("You can continue manually once it appears, or check the BQ console.")
else:
    print(f"\nLog table ready: {log_table.table_id}")

Waiting for prediction logs to appear in BigQuery...
(This can take 5-10 minutes. Logs are batched before writing.)

Found log table: carty-470812:prediction_logs.request_response_logging

Log table ready: request_response_logging


In [21]:
# Query the prediction logs to verify traffic split
# The log table includes a 'deployed_model_id' column — this is our key

if log_table:
    log_table_id = f"{PROJECT_ID}.{BQ_LOG_DATASET}.{log_table.table_id}"
    
    # First, check what columns are available
    schema_query = f"""
    SELECT column_name, data_type 
    FROM `{PROJECT_ID}.{BQ_LOG_DATASET}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{log_table.table_id}'
    """
    schema_df = bq_client.query(schema_query).to_dataframe()
    print("Log table columns:")
    for _, row in schema_df.iterrows():
        print(f"  {row['column_name']}: {row['data_type']}")
else:
    print("No log table found yet — skip to next cell if needed.")

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv-tf/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Log table columns:
  endpoint: STRING
  deployed_model_id: STRING
  logging_time: TIMESTAMP
  request_id: NUMERIC
  request_payload: ARRAY<STRING>
  response_payload: ARRAY<STRING>


In [22]:
# Analyze traffic distribution by deployed model
if log_table:
    traffic_query = f"""
    SELECT 
        deployed_model_id,
        COUNT(*) as request_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as traffic_pct
    FROM `{log_table_id}`
    GROUP BY deployed_model_id
    ORDER BY request_count DESC
    """
    
    traffic_df = bq_client.query(traffic_query).to_dataframe()
    
    # Map deployed model IDs to names
    id_to_name = {
        champion_deployed_id: "Champion (target: 80%)",
        challenger_deployed_id: "Challenger (target: 20%)"
    }
    traffic_df["model_name"] = traffic_df["deployed_model_id"].map(id_to_name)
    
    print("=== Traffic Split Verification ===")
    print(traffic_df[["model_name", "request_count", "traffic_pct"]].to_string(index=False))
    print(f"\nTotal logged requests: {traffic_df['request_count'].sum()}")
    print("\nNote: Actual percentages may not perfectly match targets.")
    print("With 150 requests, expect some variance (normal with small samples).")
else:
    print("Log table not available — check BigQuery console.")

=== Traffic Split Verification ===
              model_name  request_count  traffic_pct
  Champion (target: 80%)            126         84.0
Challenger (target: 20%)             24         16.0

Total logged requests: 150

Note: Actual percentages may not perfectly match targets.
With 150 requests, expect some variance (normal with small samples).


In [23]:
# Sample some individual log entries to see the full structure
if log_table:
    sample_query = f"""
    SELECT *
    FROM `{log_table_id}`
    LIMIT 5
    """
    sample_df = bq_client.query(sample_query).to_dataframe()
    print("Sample log entries:")
    print(sample_df.to_string())

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv-tf/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Sample log entries:
                                                                    endpoint    deployed_model_id                     logging_time                     request_id                      request_payload response_payload
0  projects/873708835509/locations/us-central1/endpoints/1246688405979398144  6624945135199191040 2026-03-15 05:40:32.292239+00:00  4217687919923911873.000000000     [[77,4,6,5,2,9,5,2,0,0,0,10,39]]              [0]
1  projects/873708835509/locations/us-central1/endpoints/1246688405979398144  6624945135199191040 2026-03-15 05:40:32.523128+00:00  7460659253208930717.000000000    [[55,4,6,5,2,13,5,4,0,0,0,23,39]]              [0]
2  projects/873708835509/locations/us-central1/endpoints/1246688405979398144  6624945135199191040 2026-03-15 05:40:33.061979+00:00  6983558633528706559.000000000     [[28,4,6,5,2,7,5,4,0,0,0,40,39]]              [0]
3  projects/873708835509/locations/us-central1/endpoints/1246688405979398144  6624945135199191040 2026-03-15 05:40:3

---
## Part 5: Canary Rollout — Shift Traffic

Now we simulate a canary rollout by progressively shifting traffic:
1. **80/20** (current) — initial canary with limited exposure
2. **50/50** — gaining confidence, equal split for comparison
3. **0/100** — full promotion of the challenger

In production you'd monitor metrics at each stage and roll back if the challenger underperforms.

In [24]:
# Helper function to update traffic and send test requests
def update_traffic_and_test(endpoint, champion_id, challenger_id, champion_pct, challenger_pct, num_requests=50):
    """Update traffic split and send test requests."""
    print(f"\n{'='*50}")
    print(f"Updating traffic split: Champion={champion_pct}%, Challenger={challenger_pct}%")
    print(f"{'='*50}")
    
    # Update the traffic split
    endpoint.update(
        traffic_split={champion_id: champion_pct, challenger_id: challenger_pct}
    )
    print("Traffic split updated.")
    
    # Short pause for the split to take effect
    time.sleep(10)
    
    # Send test requests
    print(f"Sending {num_requests} test requests...")
    for i in range(num_requests):
        instance = test_instances[i % len(test_instances)]
        endpoint.predict(instances=[instance])
    
    print(f"Done. Requests will appear in BQ logs after a short delay.")
    return datetime.utcnow()

In [25]:
# Stage 2: 50/50 split
stage2_time = update_traffic_and_test(
    endpoint, champion_deployed_id, challenger_deployed_id,
    champion_pct=50, challenger_pct=50,
    num_requests=100
)


Updating traffic split: Champion=50%, Challenger=50%
Updating Endpoint endpoint: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Endpoint endpoint updated. Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Traffic split updated.
Sending 100 test requests...
Done. Requests will appear in BQ logs after a short delay.


In [26]:
# Stage 3: Full promotion — 0/100 (challenger takes all traffic)
stage3_time = update_traffic_and_test(
    endpoint, champion_deployed_id, challenger_deployed_id,
    champion_pct=0, challenger_pct=100,
    num_requests=50
)


Updating traffic split: Champion=0%, Challenger=100%
Updating Endpoint endpoint: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Endpoint endpoint updated. Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Traffic split updated.
Sending 50 test requests...
Done. Requests will appear in BQ logs after a short delay.


In [27]:
# Verify the final configuration
endpoint = aiplatform.Endpoint(endpoint.resource_name)  # Refresh
print("=== Final Traffic Split ===")
print(f"  {endpoint.gca_resource.traffic_split}")
print("\nChallenger is now the sole model receiving traffic.")
print("In production, you'd now undeploy the champion to free resources.")

=== Final Traffic Split ===
  {'3685924165874876416': 100, '6624945135199191040': 0}

Challenger is now the sole model receiving traffic.
In production, you'd now undeploy the champion to free resources.


In [29]:
# Final log analysis — overall traffic distribution across all stages
# Wait a bit for the last batch of logs
print("Waiting 2 minutes for final logs to flush to BigQuery...")
time.sleep(120)

if log_table:
    final_query = f"""
    SELECT 
        deployed_model_id,
        COUNT(*) as total_requests,
        MIN(logging_time) as first_request,
        MAX(logging_time) as last_request
    FROM `{log_table_id}`
    GROUP BY deployed_model_id
    ORDER BY total_requests DESC
    """
    
    final_df = bq_client.query(final_query).to_dataframe()
    final_df["model_name"] = final_df["deployed_model_id"].map(id_to_name)
    
    print("=== Overall Traffic Distribution (All Stages) ===")
    print(final_df.to_string(index=False))
    print(f"\nTotal requests logged: {final_df['total_requests'].sum()}")
    print("\nThe champion should have more total requests (it had higher allocation")
    print("in the first two stages), while the challenger caught up in stage 3.")

Waiting 2 minutes for final logs to flush to BigQuery...


/Users/james.carty/Documents/VScode/google-ml-engineer/.venv-tf/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


=== Overall Traffic Distribution (All Stages) ===
  deployed_model_id  total_requests                    first_request                     last_request               model_name
6624945135199191040             243 2026-03-15 05:40:32.292239+00:00 2026-03-15 05:44:46.573665+00:00   Champion (target: 80%)
3685924165874876416              57 2026-03-15 05:40:34.171017+00:00 2026-03-15 05:44:46.833312+00:00 Challenger (target: 20%)

Total requests logged: 300

The champion should have more total requests (it had higher allocation
in the first two stages), while the challenger caught up in stage 3.


---
## Part 6: Deployment Strategy Decision Framework

### Canary Deployment

**What:** Route a small percentage of live traffic to the new model, monitor, gradually increase.

**When to use:**
- You have a new model version and want to limit blast radius
- You can monitor production metrics in near-real-time
- You want gradual, controlled rollout

**GCP mechanism:** Vertex AI Endpoint `traffic_split` — deploy both models to one endpoint, set percentages.

**Key characteristics:**
- Users DO see the new model's predictions
- Rollback = shift traffic back to 100% champion
- Simple to implement on Vertex AI

**Exam pattern:** _"You want to gradually roll out a new model while minimizing risk"_ → Canary

---

### A/B Testing

**What:** Split traffic between two models with the goal of statistically measuring which performs better on a business metric.

**When to use:**
- You need statistical proof that Model B > Model A on a specific metric
- You can track end-to-end outcomes (e.g., click-through rate, conversion)
- You need to run the test long enough for statistical significance

**GCP mechanism:** Same as canary — `traffic_split` on the endpoint. The difference is in the intent and measurement, not the infrastructure.

**Key characteristics:**
- Users DO see both models' predictions
- Requires tracking which user got which model (logging)
- Needs sufficient sample size for statistical significance
- Typically runs longer than a canary

**Exam pattern:** _"You need to determine which model drives better business outcomes"_ → A/B test

**Canary vs A/B:** Canary is about safe rollout (operational). A/B is about statistical comparison (analytical). Same mechanism, different purpose.

---

### Shadow Deployment (Dark Launch)

**What:** Send 100% of traffic to the champion for real responses. Simultaneously send the same requests to the challenger, but discard its predictions. Compare offline.

**When to use:**
- The new model is high-risk (e.g., medical, financial) and you can't expose users to untested predictions
- You want to compare prediction distributions without any user impact
- You need to validate the challenger handles production traffic patterns (latency, error rates)

**GCP mechanism:** NOT built into Vertex AI natively. You'd build this with:
- A Cloud Function / Cloud Run proxy that fans out requests to both endpoints
- Log challenger predictions to BigQuery for offline analysis
- Only return the champion's response to the user

**Key characteristics:**
- Users NEVER see the challenger's predictions
- Doubles your serving infrastructure cost
- No risk to users, but no real-world outcome measurement either

**Exam pattern:** _"You need to validate a new model in production without affecting users"_ → Shadow deployment

---

### Quick Decision Matrix

| Question | Canary | A/B Test | Shadow |
|----------|--------|----------|--------|
| Users see new model? | Yes (small %) | Yes (split %) | No |
| Goal | Safe rollout | Statistical comparison | Risk-free validation |
| Vertex AI native? | ✅ traffic_split | ✅ traffic_split | ❌ Custom proxy |
| Duration | Short (hours/days) | Medium (days/weeks) | Medium (days) |
| Cost | Low (same endpoint) | Low (same endpoint) | High (2x serving) |
| Rollback | Shift traffic | Shift traffic | Just stop forwarding |
| When exam says... | "minimize risk during rollout" | "measure business impact" | "validate without user impact" |

---
## Part 7: Cleanup

⚠️ **Run this section to avoid ongoing charges.** Deployed models on endpoints incur costs.

In [31]:
# Undeploy all models — must undeploy 0% traffic models first
endpoint = aiplatform.Endpoint(endpoint.resource_name)  # Refresh

# Sort: undeploy 0% traffic models first, then the rest
traffic = endpoint.gca_resource.traffic_split
deployed = sorted(
    endpoint.gca_resource.deployed_models,
    key=lambda dm: traffic.get(dm.id, 0)  # 0% first
)

for dm in deployed:
    print(f"Undeploying {dm.display_name} (ID: {dm.id}, traffic: {traffic.get(dm.id, 0)}%)...")
    endpoint.undeploy(deployed_model_id=dm.id)
    print(f"  Undeployed.")

print("\nAll models undeployed.")

Undeploying champion (ID: 6624945135199191040, traffic: 0%)...
Undeploying Endpoint model: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Undeploy Endpoint model backing LRO: projects/873708835509/locations/us-central1/endpoints/1246688405979398144/operations/8204395315076268032
Endpoint model undeployed. Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
  Undeployed.
Undeploying challenger (ID: 3685924165874876416, traffic: 100%)...
Undeploying Endpoint model: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Undeploy Endpoint model backing LRO: projects/873708835509/locations/us-central1/endpoints/1246688405979398144/operations/3446166386898894848
Endpoint model undeployed. Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
  Undeployed.

All models undeployed.


In [32]:
# Delete the endpoint
endpoint.delete()
print("Endpoint deleted.")

Deleting Endpoint : projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Endpoint deleted. . Resource name: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Deleting Endpoint resource: projects/873708835509/locations/us-central1/endpoints/1246688405979398144
Delete Endpoint backing LRO: projects/873708835509/locations/us-central1/operations/8210693317680168960
Endpoint resource projects/873708835509/locations/us-central1/endpoints/1246688405979398144 deleted.
Endpoint deleted.


In [33]:
# Delete the model resources
champion_vertex_model.delete()
print("Champion model deleted.")

challenger_vertex_model.delete()
print("Challenger model deleted.")

Deleting Model : projects/873708835509/locations/us-central1/models/4036895424286556160
Model deleted. . Resource name: projects/873708835509/locations/us-central1/models/4036895424286556160
Deleting Model resource: projects/873708835509/locations/us-central1/models/4036895424286556160
Delete Model backing LRO: projects/873708835509/locations/us-central1/models/4036895424286556160/operations/3861623452523823104
Model resource projects/873708835509/locations/us-central1/models/4036895424286556160 deleted.
Champion model deleted.
Deleting Model : projects/873708835509/locations/us-central1/models/4745086465690566656
Model deleted. . Resource name: projects/873708835509/locations/us-central1/models/4745086465690566656
Deleting Model resource: projects/873708835509/locations/us-central1/models/4745086465690566656
Delete Model backing LRO: projects/873708835509/locations/us-central1/models/4745086465690566656/operations/878023883762958336
Model resource projects/873708835509/locations/us-ce

In [34]:
# Clean up GCS artifacts
blobs = bucket.list_blobs(prefix=f"{LAB_PREFIX}/")
for blob in blobs:
    blob.delete()
    print(f"Deleted gs://{BUCKET_NAME}/{blob.name}")
print("GCS artifacts cleaned up.")

Deleted gs://carty-470812/mini-lab-d/
Deleted gs://carty-470812/mini-lab-d/models/
Deleted gs://carty-470812/mini-lab-d/models/challenger/
Deleted gs://carty-470812/mini-lab-d/models/challenger/model.pkl
Deleted gs://carty-470812/mini-lab-d/models/champion/model.pkl
GCS artifacts cleaned up.


In [35]:
# Optionally delete the BQ prediction logs dataset
# Uncomment to delete:
bq_client.delete_dataset(f"{PROJECT_ID}.{BQ_LOG_DATASET}", delete_contents=True)
print(f"Deleted BQ dataset: {BQ_LOG_DATASET}")

print("\n=== Cleanup Complete ===")
print("Note: BQ prediction_logs dataset retained for review.")
print("Uncomment the delete line above to remove it.")

Deleted BQ dataset: prediction_logs

=== Cleanup Complete ===
Note: BQ prediction_logs dataset retained for review.
Uncomment the delete line above to remove it.


---
## Key Takeaways

1. **Traffic splitting is Vertex AI's native deployment strategy mechanism** — one endpoint, multiple deployed models, percentage-based routing.

2. **Canary and A/B use the same infrastructure (`traffic_split`)** — the difference is intent: canary = safe rollout, A/B = statistical comparison.

3. **Shadow deployment requires custom infrastructure** — Vertex AI doesn't support it natively; you'd build a proxy layer.

4. **Prediction request-response logging to BigQuery** gives visibility into which model served each request — essential for A/B analysis.

5. **Gradual rollout pattern:** Start small (e.g., 5-10%), monitor, increase, promote. Rollback = shift traffic back.

6. **Exam keywords:** "minimize risk" → canary, "measure business impact" → A/B test, "validate without affecting users" → shadow.